# Notebook 03 - Pipeline Integration Demo

Chạy cả 2 pipeline (U²-Net + YOLO) trên ảnh dummy.

In [ ]:
import os, sys
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
sys.path.insert(0, '..')
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ml2.pipeline_integration.pipeline_u2net import warp_to_rect, enhance

img_path = sorted(Path('../data/dummy/images').glob('*.jpg'))[0]
img = cv2.imread(str(img_path))
mask = cv2.imread(str(Path('../data/dummy/masks') / (img_path.stem + '.png')), cv2.IMREAD_GRAYSCALE)
print(f'Input: {img.shape}, Mask: {mask.shape}')

In [ ]:
import numpy as np
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnt = max(contours, key=cv2.contourArea)
peri = cv2.arcLength(cnt, True)
approx = cv2.approxPolyDP(cnt, 0.02 * peri, True).squeeze(1).astype(np.float32)
warped = warp_to_rect(img, approx)
enhanced = enhance(warped)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('Input')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Mask (GT)')
axes[2].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)); axes[2].set_title('Warped')
axes[3].imshow(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)); axes[3].set_title('Enhanced')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()